# Exploratory Data Analysis: Netflix Prize Dataset

Let's get our environment set up and load in the processed data. We'll use standard visualization tools to get a feel for the dataset before we start modeling. This data is fascinating because of its sheer scale!

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Add root dir so we can import src or config
sys.path.append(os.path.abspath('..'))
import config
from src.data_loader import load_processed, load_movie_titles

# Load our data (this might take a minute depending on your disk speed!)
data_path = os.path.join('..', config.PROCESSED_DATA_DIR, 'all_ratings.parquet')
titles_path = os.path.join('..', config.RAW_DATA_DIR, 'movie_titles.txt')

# df = pd.read_csv(data_path)  # Too slow! We use parquet instead.
df = load_processed(data_path)
titles = load_movie_titles(titles_path)

## Dataset Overview

This is the famous Netflix Prize dataset. It contains nearly 100M ratings from roughly 480K users across 17.7K movies. Let's take a quick look at the shape, data types, and basic statistics.

In [ ]:
print(f"Dataset shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)

mem_usage = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemory usage: {mem_usage:.2f} MB")

display(df.head(10))

# Let's check summary statistics for the rating column
display(df[['rating']].describe())

## Rating Distribution

How generous are users with their ratings? Let's look at the histogram. We expect it to be skewed toward higher ratings since people tend to rate things they actually enjoy.

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='rating', palette='viridis')

total = len(df)
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + total*0.01,
            f'{height/1e6:.1f}M\n({height/total*100:.1f}%)',
            ha="center", va='bottom', fontsize=10)

plt.title("Distribution of Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.ylim(0, df['rating'].value_counts().max() * 1.2) # Give space for annotations
plt.show()

mean_rtg = df['rating'].mean()
med_rtg = df['rating'].median()
mode_rtg = df['rating'].mode()[0]
std_rtg = df['rating'].std()

print(f"Mean Rating:   {mean_rtg:.3f}")
print(f"Median Rating: {med_rtg}")
print(f"Mode Rating:   {mode_rtg}")
print(f"Std Dev:       {std_rtg:.3f}")

Observation: The distribution is heavily skewed toward 3-5 stars, with 4 being the most common rating. People rarely give 1 or 2 stars compared to 4s and 5s.

## User Activity Analysis

Next, we want to understand how active our users are. Are a few hardcore cinephiles generating all the ratings, or is activity more evenly spread out?

In [ ]:
user_counts = df['user_id'].value_counts()

plt.figure(figsize=(10, 5))
sns.histplot(user_counts.values, bins=50, log_scale=(True, False))
plt.title("Distribution of Ratings Per User (Log Scale)")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Users")
plt.show()

# Calculate percentiles
percentiles = np.percentile(user_counts.values, [25, 50, 75, 90, 99])
print("User Activity Percentiles:")
print(f"25th: {percentiles[0]:.0f}")
print(f"50th: {percentiles[1]:.0f} (Median)")
print(f"75th: {percentiles[2]:.0f}")
print(f"90th: {percentiles[3]:.0f}")
print(f"99th: {percentiles[4]:.0f}\n")

# Let's visualize the 80/20 rule: what % of top users account for what % of ratings?
# user_counts_sorted = np.sort(user_counts.values)[::-1]
# cum_ratings = np.cumsum(user_counts_sorted) / total
user_counts_sorted = user_counts.sort_values(ascending=False)
cum_ratings = user_counts_sorted.cumsum() / len(df) * 100
cum_users = np.arange(1, len(user_counts) + 1) / len(user_counts) * 100

plt.figure(figsize=(8, 5))
plt.plot(cum_users, cum_ratings, color='firebrick')
plt.fill_between(cum_users, cum_ratings, alpha=0.2, color='firebrick')
plt.title("Cumulative User Activity (The 80/20 Rule Check)")
plt.xlabel("% of Total Users (sorted by activity)")
plt.ylabel("% of Total Ratings")
plt.plot([0, 100], [0, 100], 'k--', alpha=0.5)
plt.show()

print(f"Most active user has {user_counts.max()} ratings.")
print(f"Median user has {user_counts.median()} ratings.\n")

print("Top 10 Most Active Users:")
print(user_counts.head(10))

## Movie Popularity Analysis

Let's look at things from the movie perspective. We'll merge with the titles dataframe to make this human-readable.

In [ ]:
movie_stats = df.groupby('movie_id').agg(
    num_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean')
)
movie_stats = movie_stats.merge(titles, on='movie_id', how='left')

# Top 20 most rated
top_20 = movie_stats.sort_values('num_ratings', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_20, x='num_ratings', y='title', palette='Blues_r')
plt.title("Top 20 Most Rated Movies")
plt.xlabel("Number of Ratings")
plt.ylabel("")
plt.show()

# Scatter plot: number of ratings vs average score
plt.figure(figsize=(10, 6))
sns.scatterplot(data=movie_stats, x='num_ratings', y='avg_rating', alpha=0.3, s=20)

# Let's annotate a few famous outliers (highly rated, very popular)
outliers = movie_stats[(movie_stats['num_ratings'] > 150000) & (movie_stats['avg_rating'] > 4.2)]
for _, row in outliers.iterrows():
    plt.text(row['num_ratings'], row['avg_rating'], row['title'], fontsize=8)

plt.title("Number of Ratings vs Average Rating Per Movie")
plt.xlabel("Number of Ratings")
plt.ylabel("Average Rating")
plt.show()

# Long tail distribution of popularity
# sns.lineplot(data=movie_stats['num_ratings'].sort_values(ascending=False).values)
plt.figure(figsize=(8, 5))
plt.plot(movie_stats['num_ratings'].sort_values(ascending=False).values)
plt.xscale('log')
plt.yscale('log')
plt.title("Movie Popularity Distribution (Log-Log Scale)")
plt.xlabel("Movie Rank")
plt.ylabel("Number of Ratings")
plt.show()

## Temporal Patterns

Let's see how rating activity changed over time. The dataset covers up to late 2005, so we should see Netflix's early growth reflected here.

In [ ]:
# Resample by month to smooth things out
monthly_stats = df.set_index('date').resample('ME').agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
)

fig, ax1 = plt.subplots(figsize=(12, 5))

color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Number of Ratings', color=color)
ax1.plot(monthly_stats.index, monthly_stats['rating_count'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Average Rating', color=color)
ax2.plot(monthly_stats.index, monthly_stats['avg_rating'], color=color, alpha=0.7)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.title("Rating Volume and Average Score Over Time")
plt.show()

# Note: Activity explodes massively around 2004-2005! Interestingly, the average rating
# drops slightly as volume goes up (perhaps the broader user base is more critical).

## Sparsity Analysis

Recommendation matrices are famously empty. Most users only ever watch a tiny fraction of available movies. Let's calculate exactly how empty our matrix is and visualize a slice of it.

In [ ]:
n_users = df['user_id'].nunique()
n_movies = df['movie_id'].nunique()
total_interactions = len(df)
possible_interactions = n_users * n_movies

sparsity = 1.0 - (total_interactions / possible_interactions)

print(f"Users: {n_users:,}")
print(f"Movies: {n_movies:,}")
print(f"Total Possible Interactions: {possible_interactions:,}")
print(f"Actual Ratings: {total_interactions:,}")
print(f"Sparsity: {sparsity * 100:.4f}%\n")

# Let's visualize a 200x200 patch of the matrix to see how sparse it looks.
# We'll grab the top users and movies so we actually see some dots.
top_users = df['user_id'].value_counts().head(200).index
top_movies = df['movie_id'].value_counts().head(200).index

sample_df = df[(df['user_id'].isin(top_users)) & (df['movie_id'].isin(top_movies))]
pivot_sample = sample_df.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)

plt.figure(figsize=(8, 8))
plt.imshow(pivot_sample.values > 0, cmap='binary', aspect='auto')
plt.title("User-Item Matrix Heatmap (Top 200 Users vs Top 200 Movies)")
plt.xlabel("Movies")
plt.ylabel("Users")
plt.show()

Even when selecting our *most active* users and *most popular* movies, the matrix is still quite sparse. This sparsity is exactly why simple memory-based collaborative filtering (like user-user KNN) struggles: it's hard to find users with enough overlapping movies to calculate reliable similarities. This motivates the use of matrix factorization.

## Key Takeaways

* **Skewed Ratings:** Users generally rate favorably; the vast majority of ratings are 3s, 4s, and 5s.
* **Power Law Distributions:** Both user activity and movie popularity follow extreme long-tail distributions. A small core group of movies and users dominate the interaction volume.
* **Temporal Growth:** The volume of ratings spiked dramatically around 2004-2005, reflecting Netflix's rapid adoption phase.
* **Extreme Sparsity:** The overall matrix is highly sparse (>98% empty). Most users haven't rated most movies.
* **Popularity Bias:** Highly rated movies tend to have more ratings overall. Niche movies have far fewer ratings, meaning less reliable signal.

**Recommendation:** 
Given the extreme sparsity and massive scale of the data, memory-based nearest-neighbors models will be computationally expensive and may suffer from lack of overlap. **Singular Value Decomposition (SVD)** or other latent factor models should be our primary modeling approach. These models can uncover underlying patterns and fill in the "blank space" much more robustly than basic KNN techniques.